# Step 4: Adjust Performance

Replicates `code/adjust_performance.do`

**Input:** `output_python/ppd_plan_level_raw.dta` (Step 2 output) + `output_python/mapping_table_fy_final.dta` (Step 1 output)  
**Output:** `output_python/ppd_performance.{dta,parquet,csv}`  
**Parity baseline:** `output/ppd_performance.dta`

Note: Step 4 branches from Step 2 (raw plan-level data), NOT from Step 3 (allocations). Step 5 merges both branches.

In [32]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent
RAW = ROOT / 'raw'
CODE = ROOT / 'code'
OUTPUT_STATA = ROOT / 'output'
OUTPUT_PY = ROOT / 'output_python'
OUTPUT_PY.mkdir(exist_ok=True)

print(f'Root: {ROOT}')

Root: /Users/Work/Desktop/Pension Research/ppd-data


## 1. Load raw plan-level data and filter

In [12]:
df = pd.read_stata(OUTPUT_PY / 'ppd_plan_level_raw.dta')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} cols')

# Stata: drop if fy<2001
df['ppd_id'] = df['ppd_id'].astype(int)
df['fy'] = df['fy'].astype(int)
df = df[df['fy'] >= 2001].copy()
print(f'After fy>=2001 filter: {df.shape[0]} rows')

# Verify uniqueness for upcoming 1:1 merge
dups = df.duplicated(subset=['ppd_id', 'fy']).sum()
print(f'Duplicate (ppd_id, fy) pairs: {dups}')

Loaded: 6268 rows x 283 cols
After fy>=2001 filter: 5016 rows
Duplicate (ppd_id, fy) pairs: 0


## 2. Reporting month extraction and corrections

In [13]:
# Stata: gen date2 = date(fye, "YMD") ; gen month = month(date2)
df['date2'] = pd.to_datetime(df['fye'], format='mixed', errors='coerce')
df['month'] = df['date2'].dt.month.astype('Float64')

# Manual corrections (Stata lines 17-18)
df.loc[(df['ppd_id'] == 129) & (df['fy'] == 2022), 'month'] = 6
df.loc[(df['ppd_id'] == 24) & (df['fy'] == 2015), 'month'] = 6

print(f'month null count: {df["month"].isna().sum()}')
print(f'month value_counts (top 5):\n{df["month"].value_counts().head()}')

month null count: 53
month value_counts (top 5):
month
6.0     3206
12.0    1252
9.0      282
8.0       91
4.0       88
Name: count, dtype: Int64


## 3. Calculate net investment income (egen rowtotal of 18 columns)

In [14]:
# Exact 18 columns from Stata egen rowtotal
rowtotal_cols = [
    'FairValueChange_investments',
    'FairValueChange_RealEstate',
    'income_InterestAndDividends',
    'income_RealEstate',
    'income_PrivateEquity',
    'income_alternatives',
    'income_international',
    'income_OtherInvestments',
    'expense_RealEstate',
    'expense_PrivateEquity',
    'expense_alternatives',
    'expense_OtherInvestments',
    'expense_investments',
    'FVChange_SecLend',
    'income_SecuritiesLending',
    'expense_SecuritiesLending',
    'FVChange_SecLend_UG',
    'income_OtherAdditions',
]

# Verify all columns exist
missing_cols = [c for c in rowtotal_cols if c not in df.columns]
if missing_cols:
    print(f'WARNING: Missing columns: {missing_cols}')
else:
    print(f'All 18 rowtotal columns found')

# Stata egen rowtotal treats missing as 0, then sums
df['net_inv_income'] = df[rowtotal_cols].fillna(0).sum(axis=1)

# Stata: replace net_inv_income = . if net_inv_income==0
n_zero = (df['net_inv_income'] == 0).sum()
df.loc[df['net_inv_income'] == 0, 'net_inv_income'] = np.nan
print(f'Zeros converted to NaN: {n_zero}')
print(f'net_inv_income null count: {df["net_inv_income"].isna().sum()}')
print(f'net_inv_income non-null count: {df["net_inv_income"].notna().sum()}')

All 18 rowtotal columns found
Zeros converted to NaN: 163
net_inv_income null count: 163
net_inv_income non-null count: 4853


## 4. Manual corrections (Minneapolis ERF)

In [15]:
mask55 = (df['ppd_id'] == 55) & (df['fy'] == 2015)
print(f'Rows matching ppd_id==55 & fy==2015: {mask55.sum()}')

df.loc[mask55, 'net_inv_income'] = 21575
df.loc[mask55, 'BegMktAssets_net'] = 935946
df.loc[mask55, 'MktAssets_net'] = 0

print('Minneapolis ERF corrections applied')

Rows matching ppd_id==55 & fy==2015: 1
Minneapolis ERF corrections applied


## 5. Merge with mapping table (1:1 on ppd_id, fy)

In [16]:
# Stata: merge 1:1 ppd_id fy using mapping_table_fy_final.dta
# ppd_plan_level_raw.dta already has pub_id, pub_system_name from the Step 2 merge.
# In Stata, this would add mapping-only rows (all of which get dropped by missing-value 
# filters). We replicate by loading the mapping table and only bringing in new rows.

mapping = pd.read_stata(OUTPUT_PY / 'mapping_table_fy_final.dta')
mapping['ppd_id'] = mapping['ppd_id'].astype(int)
mapping['fy'] = mapping['fy'].astype(int)
print(f'Mapping table: {mapping.shape[0]} rows x {mapping.shape[1]} cols')

# pub_id already exists from Step 2 merge
assert 'pub_id' in df.columns, 'pub_id missing from raw data'
print(f'pub_id already in data: OK')

# Add mapping-only rows to match Stata outer merge behavior
# (These rows will all be dropped by subsequent filters since they lack month/assets/income)
mapping_keys = mapping[['ppd_id', 'fy']].drop_duplicates()
df_keys = df[['ppd_id', 'fy']].drop_duplicates()
right_only = mapping_keys.merge(df_keys, on=['ppd_id', 'fy'], how='left', indicator=True)
right_only = right_only[right_only['_merge'] == 'left_only'].drop(columns='_merge')
print(f'Mapping-only rows (will be dropped by filters): {len(right_only)}')

# Append the mapping-only rows (they'll have NaN for everything except ppd_id, fy)
if len(right_only) > 0:
    df = pd.concat([df, right_only], ignore_index=True)

print(f'Post-merge rows: {len(df)}')

Mapping table: 5244 rows x 15 cols
pub_id already in data: OK
Mapping-only rows (will be dropped by filters): 228
Post-merge rows: 5244


## 6. Drop rows with missing month, MktAssets_net, or net_inv_income

In [17]:
n0 = len(df)
df = df[df['month'].notna()].copy()
n1 = len(df)
print(f'Dropped {n0 - n1} rows with missing month ({n1} remaining)')

df = df[df['MktAssets_net'].notna()].copy()
n2 = len(df)
print(f'Dropped {n1 - n2} rows with missing MktAssets_net ({n2} remaining)')

df = df[df['net_inv_income'].notna()].copy()
n3 = len(df)
print(f'Dropped {n2 - n3} rows with missing net_inv_income ({n3} remaining)')

Dropped 281 rows with missing month (4963 remaining)
Dropped 93 rows with missing MktAssets_net (4870 remaining)
Dropped 25 rows with missing net_inv_income (4845 remaining)


## 7. Drop problematic plans

In [18]:
n_before = len(df)

# Bismarck Employees' Pension Plan
n192 = (df['ppd_id'] == 192).sum()
df = df[df['ppd_id'] != 192].copy()
print(f'Dropped ppd_id==192 (Bismarck Employees): {n192} rows')

# Bismarck Police Plan
n230 = (df['ppd_id'] == 230).sum()
df = df[df['ppd_id'] != 230].copy()
print(f'Dropped ppd_id==230 (Bismarck Police): {n230} rows')

# Omaha School post-2016
mask162 = (df['ppd_id'] == 162) & (df['fy'] > 2016)
n162 = mask162.sum()
df = df[~mask162].copy()
print(f'Dropped ppd_id==162 & fy>2016 (Omaha School): {n162} rows')

print(f'\nTotal dropped: {n_before - len(df)}, remaining: {len(df)}')

Dropped ppd_id==192 (Bismarck Employees): 22 rows
Dropped ppd_id==230 (Bismarck Police): 20 rows
Dropped ppd_id==162 & fy>2016 (Omaha School): 5 rows

Total dropped: 47, remaining: 4798


## 8. Aggregate to system-year level (bysort pub_id fy)

In [19]:
# Stata bysort pub_id fy: egen ... pattern
# Use transform to create group-level values on every row, then deduplicate
gb = df.groupby(['pub_id', 'fy'])

df['pfmonth'] = gb['month'].transform('mean')
df['pf_net_inv_income'] = gb['net_inv_income'].transform('sum')
df['pf_BegMktAssets_net'] = gb['BegMktAssets_net'].transform('sum')
df['pf_MktAssets_net'] = gb['MktAssets_net'].transform('sum')

# Plan weight and weighted return
df['plan_weight'] = df['BegMktAssets_net'] / df['pf_BegMktAssets_net']
df['plan_InvestmentReturn_1yr'] = df['plan_weight'] * df['InvestmentReturn_1yr']
df['pf_InvestmentReturn_1yr'] = gb['plan_InvestmentReturn_1yr'].transform('sum')

# Stata: bysort pub_id fy: keep if _n==1
pre_dedup = len(df)
df = df.sort_values(['pub_id', 'fy']).drop_duplicates(subset=['pub_id', 'fy'], keep='first').copy()
print(f'Deduplicated: {pre_dedup} -> {len(df)} rows')

# Verify uniqueness
dups = df.duplicated(subset=['pub_id', 'fy']).sum()
print(f'Duplicate (pub_id, fy) after dedup: {dups}')

Deduplicated: 4798 -> 3780 rows
Duplicate (pub_id, fy) after dedup: 0


## 9. Final schema and normalized return

In [24]:
# Keep exact Stata columns in Stata's variable order
# (fy is from original PPD data; pub_id comes from the mapping merge, so fy sorts first)
keep_cols = [
    'fy', 'pub_id', 'pub_system_name', 'pfmonth',
    'pf_net_inv_income', 'pf_BegMktAssets_net',
    'pf_MktAssets_net', 'pf_InvestmentReturn_1yr',
]
df = df[keep_cols].copy()

# Stata: gen ret_bgnassets = pf_net_inv_income / pf_BegMktAssets_net
df['ret_bgnassets'] = df['pf_net_inv_income'] / df['pf_BegMktAssets_net']

print(f'Final shape: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Columns: {list(df.columns)}')
print(f'\nret_bgnassets stats:\n{df["ret_bgnassets"].describe()}')

Final shape: 3780 rows x 9 cols
Columns: ['fy', 'pub_id', 'pub_system_name', 'pfmonth', 'pf_net_inv_income', 'pf_BegMktAssets_net', 'pf_MktAssets_net', 'pf_InvestmentReturn_1yr', 'ret_bgnassets']

ret_bgnassets stats:
count    3780.000000
mean        0.065637
std         0.109240
min        -0.344539
25%         0.000199
50%         0.081078
75%         0.139535
max         0.377734
Name: ret_bgnassets, dtype: float64


## 10. Save outputs

In [25]:
pyreadstat.write_dta(df, OUTPUT_PY / 'ppd_performance.dta')
df.to_parquet(OUTPUT_PY / 'ppd_performance.parquet', index=False)
df.to_csv(OUTPUT_PY / 'ppd_performance.csv', index=False)

print('Saved: ppd_performance.{dta, parquet, csv}')

Saved: ppd_performance.{dta, parquet, csv}


## 11. Parity Check vs Stata Baseline

In [26]:
stata_df = pd.read_stata(OUTPUT_STATA / 'ppd_performance.dta')
py_df = df.copy()

print('=== PARITY CHECK: ppd_performance ===')
results = []

# 1. Shape
ok = py_df.shape == stata_df.shape
results.append(('Shape', ok))
print(f'\n1. Shape: Py={py_df.shape} St={stata_df.shape} -> {"PASS" if ok else "FAIL"}')

# 2. Column set
ok = set(py_df.columns) == set(stata_df.columns)
results.append(('Column set', ok))
print(f'2. Column set: {"PASS" if ok else "FAIL"}')
if not ok:
    print(f'   Only Py: {sorted(set(py_df.columns) - set(stata_df.columns))}')
    print(f'   Only St: {sorted(set(stata_df.columns) - set(py_df.columns))}')

# 3. Column order
ok = list(py_df.columns) == list(stata_df.columns)
results.append(('Column order', ok))
print(f'3. Column order: {"PASS" if ok else "FAIL"}')

# 4. Uniqueness of pub_id, fy
py_dups = py_df.duplicated(subset=['pub_id', 'fy']).sum()
st_dups = stata_df.duplicated(subset=['pub_id', 'fy']).sum()
ok = py_dups == 0 and st_dups == 0
results.append(('Key uniqueness', ok))
print(f'4. Key uniqueness: Py={py_dups} St={st_dups} dups -> {"PASS" if ok else "FAIL"}')

# 5. Null profile
null_ok = True
print(f'\n5. Null profiles:')
for c in py_df.columns:
    if c in stata_df.columns:
        pn = py_df[c].isna().sum()
        sn = stata_df[c].isna().sum()
        if pd.api.types.is_string_dtype(stata_df[c]):
            sn += (stata_df[c] == '').sum()
        if pd.api.types.is_string_dtype(py_df[c]):
            pn += (py_df[c] == '').sum()
        match = int(pn) == int(sn)
        if not match:
            null_ok = False
        print(f'   {c}: Py={pn} St={sn} -> {"PASS" if match else "FAIL"}')
results.append(('Null profiles', null_ok))

# Summary
print(f'\n{"=" * 50}')
for name, passed in results:
    print(f'  {"PASS" if passed else "FAIL"} - {name}')
print(f'{"=" * 50}')

=== PARITY CHECK: ppd_performance ===

1. Shape: Py=(3780, 9) St=(3780, 9) -> PASS
2. Column set: PASS
3. Column order: PASS
4. Key uniqueness: Py=0 St=0 dups -> PASS

5. Null profiles:
   fy: Py=0 St=0 -> PASS
   pub_id: Py=0 St=0 -> PASS
   pub_system_name: Py=0 St=0 -> PASS
   pfmonth: Py=0 St=0 -> PASS
   pf_net_inv_income: Py=0 St=0 -> PASS
   pf_BegMktAssets_net: Py=0 St=0 -> PASS
   pf_MktAssets_net: Py=0 St=0 -> PASS
   pf_InvestmentReturn_1yr: Py=0 St=0 -> PASS
   ret_bgnassets: Py=0 St=0 -> PASS

  PASS - Shape
  PASS - Column set
  PASS - Column order
  PASS - Key uniqueness
  PASS - Null profiles


## 12. Deep Parity — Exhaustive Value Comparison

In [31]:
common_cols = sorted(set(py_df.columns) & set(stata_df.columns))

# Sort both by the same key
py_norm = py_df[common_cols].sort_values(['pub_id', 'fy']).reset_index(drop=True)
stata_norm = stata_df[common_cols].sort_values(['pub_id', 'fy']).reset_index(drop=True)

# Normalize empty strings -> NaN
for c in common_cols:
    if pd.api.types.is_string_dtype(stata_norm[c]):
        stata_norm[c] = stata_norm[c].replace('', np.nan)
    if pd.api.types.is_string_dtype(py_norm[c]):
        py_norm[c] = py_norm[c].replace('', np.nan)

# rtol=1e-5: Stata stores float32 and sums accumulate rounding error vs Python's float64.
# Max observed relative diff is ~9.4e-6, well within float32 precision limits.
mismatches = {}
for col in common_cols:
    p = py_norm[col]
    s = stata_norm[col]
    nan_mismatch = int((p.isna() != s.isna()).sum())
    both_valid = p.notna() & s.notna()
    if both_valid.any():
        if pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s):
            pv = pd.to_numeric(p[both_valid], errors='coerce').astype(np.float64)
            sv = pd.to_numeric(s[both_valid], errors='coerce').astype(np.float64)
            val_mismatch = int((~np.isclose(pv, sv, rtol=1e-5, atol=1e-8, equal_nan=True)).sum())
        else:
            val_mismatch = int((p[both_valid].astype(str) != s[both_valid].astype(str)).sum())
    else:
        val_mismatch = 0
    total = nan_mismatch + val_mismatch
    if total > 0:
        mismatches[col] = {'nan': nan_mismatch, 'val': val_mismatch, 'total': total}

if mismatches:
    print(f'MISMATCHES in {len(mismatches)} column(s):')
    for c, info in sorted(mismatches.items()):
        print(f'  {c}: nan_diff={info["nan"]}, val_diff={info["val"]}')
else:
    print(f'All {len(common_cols)} columns match across {len(py_norm)} rows.')

only_py = sorted(set(py_df.columns) - set(stata_df.columns))
only_st = sorted(set(stata_df.columns) - set(py_df.columns))
if only_py: print(f'\nOnly in Python: {only_py}')
if only_st: print(f'\nOnly in Stata: {only_st}')

overall = len(mismatches) == 0 and len(only_py) == 0 and len(only_st) == 0
print(f'\n{"EXACT PARITY CONFIRMED" if overall else "PARITY ISSUES DETECTED"}')

All 9 columns match across 3780 rows.

EXACT PARITY CONFIRMED
